In [1]:
import os

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="2"

In [2]:
from setproctitle import setproctitle
setproctitle("ppo_zero_entropy_vs_ppo")

In [3]:
import sys
sys.path.append('..')

In [4]:
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
  tf.config.experimental.set_memory_growth(gpu, True)

2025-05-24 16:34:14.553988: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-24 16:34:14.554022: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-24 16:34:14.555319: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-24 16:34:14.561430: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-24 16:34:15.094232: W tensorflow/compiler/tf2

In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
import numpy as np
from tqdm import trange
from UltimateTicTacToeEnvSelfPlay import UltimateTicTacToeEnvSelfPlay
from A2CAgent import A2CAgent
from PPOAgent import PPOAgent

In [7]:
NUM_OF_GAMES = 500
envs = [UltimateTicTacToeEnvSelfPlay() for _ in range(NUM_OF_GAMES)]
state_space_shape = envs[0].to_state()[0].shape[0]
action_space_size = 81
a2c_agent = PPOAgent(state_space_shape, action_space_size, loaded=True, model_path="../models/Thesis_PPO_resnet")
ppo_agent = PPOAgent(state_space_shape, action_space_size, loaded=True, model_path="../models/Thesis_PPO")
dones = np.zeros((NUM_OF_GAMES,), dtype=bool)
final_rewards = np.zeros((NUM_OF_GAMES,))
while(not np.all(dones) != 0):
    print(np.sum(dones), " out of ", len(dones))
    a2c_states = np.array([env.to_state()[0] for env in envs])
    a2c_available_actions = np.array([env.to_state()[1] for env in envs])
    a2c_actions = a2c_agent.act(a2c_states, a2c_available_actions)
    a2c_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
        _, a2c_reward[i], game_finished[i], _  = envs[i].step(a2c_actions[i])
        if game_finished[i] == True and dones[i] == False:
            dones[i] = True
            final_rewards[i] = a2c_reward[i]
    
    ppo_states = np.array([env.to_state()[0] for env in envs])
    ppo_available_actions = np.array([env.to_state()[1] for env in envs])
    ppo_actions = ppo_agent.act(ppo_states, ppo_available_actions)
    ppo_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
            _, ppo_reward[i], game_finished[i], _  = envs[i].step(ppo_actions[i])
            if game_finished[i] == True and dones[i] == False:
                dones[i] = True
                final_rewards[i] = -ppo_reward[i]
    print("Both players have done a move.")

win_as_first_player = np.count_nonzero(final_rewards == 1)/len(dones)
draw_as_first_player = np.count_nonzero(final_rewards == 0)/len(dones)

print("win rate as first player: ", win_as_first_player)
print("draw rate as first player: ", draw_as_first_player)

Models loaded from memory


2025-05-24 16:34:16.263073: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-05-24 16:34:16.263386: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-05-24 16:34:16.263626: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Models loaded from memory
0  out of  500


2025-05-24 16:34:17.627422: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
1  out of  500
Both players have done a move.
3  out of  500
Both players have done a move.
3  out of  500
Both players have done a move.
4  out of  500
Both players have done a move.
4  out of  500
Both players have done a move.
7  out of  500
Both players have done a move.
15  out of  500
Both players have done a move.
25  out of  500
Both players have done a move.
48  out of  500
Both players have done a move.
91  out of  500
Both players have done a move.

In [8]:
NUM_OF_GAMES = 500
envs = [UltimateTicTacToeEnvSelfPlay() for _ in range(NUM_OF_GAMES)]
state_space_shape = envs[0].to_state()[0].shape[0]
action_space_size = 81
a2c_agent = PPOAgent(state_space_shape, action_space_size, loaded=True, model_path="../models/Thesis_PPO_resnet")
ppo_agent = PPOAgent(state_space_shape, action_space_size, loaded=True, model_path="../models/Thesis_PPO")
dones = np.zeros((NUM_OF_GAMES,), dtype=bool)
final_rewards = np.zeros((NUM_OF_GAMES,))
while(not np.all(dones) != 0):
    print(np.sum(dones), " out of ", len(dones))
    ppo_states = np.array([env.to_state()[0] for env in envs])
    ppo_available_actions = np.array([env.to_state()[1] for env in envs])
    ppo_actions = ppo_agent.act(ppo_states, ppo_available_actions)
    ppo_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
            _, ppo_reward[i], game_finished[i], _  = envs[i].step(ppo_actions[i])
            if game_finished[i] == True and dones[i] == False:
                dones[i] = True
                final_rewards[i] = -ppo_reward[i]
                
    a2c_states = np.array([env.to_state()[0] for env in envs])
    a2c_available_actions = np.array([env.to_state()[1] for env in envs])
    a2c_actions = a2c_agent.act(a2c_states, a2c_available_actions)
    a2c_reward = np.zeros(NUM_OF_GAMES)
    game_finished = np.zeros(NUM_OF_GAMES)
    for i in range(NUM_OF_GAMES):
        _, a2c_reward[i], game_finished[i], _  = envs[i].step(a2c_actions[i])
        if game_finished[i] == True and dones[i] == False:
            dones[i] = True
            final_rewards[i] = a2c_reward[i]
    print("Both players have done a move.")

win_as_second_player = np.count_nonzero(final_rewards == 1)/len(dones)
draw_as_second_player = np.count_nonzero(final_rewards == 0)/len(dones)

print("win rate as second player: ", win_as_second_player)
print("draw rate as second player: ", draw_as_second_player)

Models loaded from memory


Models loaded from memory
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
0  out of  500
Both players have done a move.
2  out of  500
Both players have done a move.
2  out of  500
Both players have done a move.
7  out of  500
Both players have done a move.
10  out of  500
Both players have done a move.
21  out of  500
Both players have done a move.
36  out of  500
Both players have done a move.
66  out of  500
Both players have done a move.
114 

In [9]:
total_win = (win_as_first_player+win_as_second_player)/2
total_draw = (draw_as_first_player+draw_as_second_player)/2
print("Total win rate: ", total_win)
print("Total draw rate: ", total_draw)
print("Total loss: ", 1-total_draw-total_win)

Total win rate:  0.484
Total draw rate:  0.053000000000000005
Total loss:  0.46299999999999997
